In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sb
import pandas as pd
from numba import jit
import xgi

sys.path.append("../code/")

from hypersync_draw import *
from hypersync_generate import *
from hypersync_identify import *
from hypersync_integrate import *


results_dir = "../results/"
data_dir = "../data/"

Path(results_dir).mkdir(parents=True, exist_ok=True)
Path(data_dir).mkdir(parents=True, exist_ok=True)
mpl.use('Qt5Agg') 
plt.ion() 

plt.rcParams['font.sans-serif'] = ['Times New Roman', 'SimHei'] 
plt.rcParams['mathtext.fontset'] = 'cm'       

In [ ]:
N = 83; nreps = 10000; suf = "di" 
file_name = "../data/thetas_arr_beta_nreps_10000.npy"
data = np.load(file_name)  
file_name = "../data/df_long_tend_800_nreps_10000_r_2.csv"
df = pd.read_csv(file_name, index_col=0)

## Local order parameter

In [ ]:
r2 = 2

@jit
def compute_local_order(phases, r2):
    n_beta, nreps, N = phases.shape

    local_order = np.zeros((n_beta, nreps, N), dtype=np.complex64)

    for ll in range(n_beta):
        for kk in range(nreps):
            psi = phases[ll, kk]

            for ii in range(N):
                for j in range(-r2, r2 + 1):
                    idx = (ii + j) % N
                    local_order[ll, kk, ii] += np.exp(1j * psi[idx])

    local_order /= 2 * r2 + 1
    local_order = np.abs(local_order)

    return local_order

In [ ]:
local_order = compute_local_order(data, r2)
threshold = 0.85
local_order_bool = local_order.copy()
local_order_bool[local_order < threshold] = 0
local_order_bool[local_order >= threshold] = 1
mean_order = np.mean(local_order_bool, axis=-1)
betas = sorted(df["beta"].unique())
y_mean = np.mean(mean_order, axis=1)
y_std = np.std(mean_order, axis=1)

In [ ]:

run_dir = f"{results_dir}run_tend_800_nreps_10000_r_2/"
file_name = f"{run_dir}df_long_tend_800_nreps_10000_r_2.csv"

df = pd.read_csv(file_name, index_col=0)

df["q"] = df["state"].apply(lambda x: int(x[:-8]) if "twisted" in x else None)
df["k"] = df["state"].apply(lambda x: int(x[:-8]) if "cluster" in x else None)
df["state"] = df["state"].replace("sync", "0-twisted")

mask_pos = (df["q"] >= 0) | (df["q"].isna())
mask_nobig = ((df["q"] > 98) | (df["q"] < 4) | (df["q"].isna()))

df_filtered = df.iloc[::2]

df_plot_left = df_filtered[mask_pos & mask_nobig].copy()

state_order = [f"{i}-twisted" for i in range(4)] + ["other"]
legend_labels = [r'$q = 0$', r'$q = 1$', r'$q = 2$', r'$q = 3$', 'Other']

betas_subset = betas[::2]
y_mean_subset = y_mean[::2]
y_std_subset = y_std[::2]

In [ ]:


def kuramoto(N, t_end, dt, r1, r2, k1, k2, initial, alpha, beta):
    """Simulates Kuramoto oscillators on a ring with higher-order interactions (modulus version)."""
    def rhs(theta, N, r1, r2, k1, k2, alpha, beta):
        dtheta_dt = np.zeros(N)
        local_order = np.zeros(N, dtype=np.complex64)
        phases = theta

        # Compute local order parameter
        for ii in range(N):
            for j in range(-r2, r2 + 1):
                idx = (ii + j) % N
                local_order[ii] += np.exp(1j * phases[idx])

        local_order /= (2 * r2 + 1)
        epsilon = 1e-10
        threshold = 1e-12
        local_order[np.abs(local_order) < threshold] = epsilon

        # Use modulus instead of complex value
        O_alpha = np.abs(local_order) ** alpha
        O_beta = np.abs(local_order) ** beta

        # Pairwise coupling
        for ii in range(N):
            for jj in range(-r1, r1 + 1):
                dtheta_dt[ii] += O_alpha[ii] * k1 * (np.sin(theta[(ii + jj) % N] - theta[ii])) / r1

        # Triadic coupling
        idx = list(range(-r2, 0)) + list(range(1, r2 + 1))
        for ii in range(N):
            for jj in idx:
                for kk in idx:
                    if jj != kk:
                        dtheta_dt[ii] += (
                            k2 * O_beta[ii]
                            * np.sin(theta[(ii + kk) % N] + theta[(ii + jj) % N] - 2 * theta[ii])
                            / (r2 * (2.0 * r2 - 1))
                        )
        return dtheta_dt, local_order, O_alpha, O_beta

    times = np.arange(0, t_end + dt / 2, dt)
    n_t = len(times)
    thetas = np.zeros((N, n_t))
    O_alphas = np.zeros((N, n_t))  
    O_betas = np.zeros((N, n_t))
    O_s = np.zeros((N, n_t))
    thetas[:, 0] = initial
    for it in range(1, n_t):
        dtheta_dt, local_order, O_alpha, O_beta = rhs(thetas[:, it - 1], N, r1, r2, k1, k2, alpha, beta)
        thetas[:, it] = thetas[:, it - 1] + dt * dtheta_dt
        O_alphas[:, it] = O_alpha
        O_betas[:, it] = O_beta
        O_s[:, it] = np.abs(local_order) 

    return thetas, O_s, O_alphas, O_betas, times


In [ ]:
def plot_phases_line(thetas, it=-1, ax=None, **kwargs):
    if ax is None:
        ax = plt.gca()

    psi = thetas[:, it]

    ax.plot(psi % (2 * np.pi), "o", **kwargs)

    sb.despine()

    ax.set_ylim([-0.1, 2 * np.pi + 0.1])
    ax.set_yticks([0, np.pi, 2 * np.pi])
    ax.set_yticklabels([0, r"$\pi$", r"$2\pi$"])
    ax.set_ylabel("Phase")

    return ax

In [ ]:
# Time Series
zz = 15
N = 83
r1 = 2
r2 = 2
k1 = 1
t_end = 800
dt = 0.01
alpha = 0


t_arr = np.arange(0, t_end + dt, dt)


param_sets = [(5, 0), (5, 0.5), (5, 1), (5, 2.0)] 

Path("./data").mkdir(parents=True, exist_ok=True)

for k2, beta in param_sets:
    H = xgi.trivial_hypergraph(N)
    psi_init = np.random.random(N) * 2 * np.pi

    thetas, O_s, O_alphas, O_betas, _ = kuramoto(
        N, t_end, dt, r1, r2, k1, k2, psi_init, alpha, beta
    )
    thetas = thetas % (2 * np.pi)

    local_order = np.zeros((len(t_arr), N))
    for t in range(len(t_arr)):
        for i in range(N):
            idxs = [(i + j) % N for j in range(-r2, r2 + 1)]
            local_order[t, i] = np.abs(np.mean(np.exp(1j * thetas[idxs, t])))

    T_final = 100 
    t_start_idx = np.searchsorted(t_arr, t_end - T_final)

    thetas_final = thetas[:, t_start_idx:]
    local_order_final = local_order[t_start_idx:]
    t_final_arr = t_arr[t_start_idx:] 


    np.save(f"../data/thetas_k2_{k2}_beta_{beta}.npy", thetas_final)
    np.save(f"../data/local_order_k2_{k2}_beta_{beta}.npy", local_order_final)
    np.save(f"../data/times_k2_{k2}_beta_{beta}.npy", t_final_arr)
    print(f"Saved: k2={k2}, beta={beta}, time ∈ [{t_final_arr[0]:.2f}, {t_end}]")

In [ ]:
import matplotlib.gridspec as gridspec

plt.close("all")
zz = 15 
param_sets = [(5, 0), (5, 0.5), (5, 1.0), (5, 2.0)]
fig = plt.figure(figsize=(11, 4.5))  
outer_gs = gridspec.GridSpec(nrows=1, ncols=2, width_ratios=[2.5, 4], wspace=0.2)  

ax_a = fig.add_subplot(outer_gs[0])
ax_a.errorbar(
    betas_subset, y_mean_subset, yerr=y_std_subset,
    marker="o", ms=5, color="k", ecolor="silver",
    mfc="k", mec="k", zorder=2,
)
ax_a.set_ylim([-0.01, 1.01])
ax_a.set_yticks([0, 0.5, 1])
ax_a.set_xticks(np.arange(0, 2.5, 0.5))
ax_a.axhline(y=0, color="gray", linestyle="--", linewidth=1, alpha=0.6)
ax_a.set_xlabel(rf"$\beta$", fontsize=18)
ax_a.set_ylabel("$P_{\mathrm{order}}$", fontsize=18)
ax_a.tick_params(axis='both', labelsize=16)
ax_a.text(0.01, 1.08, "(a)", transform=ax_a.transAxes, fontsize=16, va="top", ha="left")
sb.despine(ax=ax_a)

inner_gs = gridspec.GridSpecFromSubplotSpec(
    3, 4, subplot_spec=outer_gs[1], wspace=0.25, hspace=0.25
)

for idx, (k2, beta) in enumerate(param_sets):
    col = idx

    thetas = np.load(f"../data/thetas_k2_{k2}_beta_{beta}.npy")
    local_order = np.load(f"../data/local_order_k2_{k2}_beta_{beta}.npy")
    times = np.load(f"../data/times_k2_{k2}_beta_{beta}.npy")
    extent = [0, N, times[0], times[-1]]

    # === 第一行：相位线图 ===
    ax0 = fig.add_subplot(inner_gs[0, col])
    plot_phases_line(thetas, -1, ax=ax0, color=(0.0, 0.45, 0.74), ms=3)
    ax0.set_xticks([])
    if col == 0:
        ax0.set_ylabel(r"$\theta_i$", fontsize=zz + 2)
        ax0.set_yticks([0, np.pi, 2 * np.pi])
        ax0.set_yticklabels([r"$0$", r"$\pi$", r"$2\pi$"], fontsize=zz + 1)
    else:
        ax0.set_yticklabels([])
        ax0.set_ylabel("")
    ax0.set_xlabel("")

    ax0.text(0.01, 1.35, f"({chr(98 + col)}) $\\beta$ = {beta}", transform=ax0.transAxes,
             fontsize=16, va="top", ha="left")

    # === 第二行：相位热图 ===
    ax1 = fig.add_subplot(inner_gs[1, col])
    im1 = ax1.imshow(thetas.T, aspect="auto", origin="lower", cmap="coolwarm",
                     extent=extent, vmin=0, vmax=2 * np.pi)
    ax1.set_xticks([])  # 不显示x轴刻度
    if col == 0:
        ax1.set_ylabel(r"$\it{Time}$", fontsize=zz)
        ax1.set_yticks([700, 750, 800])
        ax1.set_yticklabels(["700", "750", "800"], fontsize=zz - 2)
    else:
        ax1.set_yticklabels([])
    if col == len(param_sets) - 1:
        cbar1 = fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
        cbar1.set_label(r"$\theta_i$", fontsize=zz)
        cbar1.set_ticks([0, np.pi, 2*np.pi])
        cbar1.set_ticklabels(['0', r'$\pi$', r'$2\pi$'])
        cbar1.ax.tick_params(labelsize=zz)

    ax2 = fig.add_subplot(inner_gs[2, col])
    im2 = ax2.imshow(local_order, aspect="auto", origin="lower", cmap="viridis",
                     extent=extent, vmin=0, vmax=1)
    ax2.set_xlabel(r"$\it{i}$", fontsize=zz + 2)
    ax2.set_xticks([0, 40, 80])
    ax2.tick_params(axis='x', labelsize=zz)
    if col == 0:
        ax2.set_ylabel(r"$\it{Time}$", fontsize=zz)
        ax2.set_yticks([700, 750, 800])
        ax2.set_yticklabels(["700", "750", "800"], fontsize=zz - 2)
    else:
        ax2.set_yticklabels([])
    if col == len(param_sets) - 1:
        cbar2 = fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
        cbar2.set_label(r"$|O_i|$", fontsize=zz)
        cbar2.set_ticks([0, 0.5, 1.0])
        cbar2.set_ticklabels(['0', '0.5', '1.0'])
        cbar2.ax.tick_params(labelsize=zz)

# plt.savefig("fig5.eps", format="eps",dpi=300, bbox_inches="tight", pad_inches=0)
plt.show()
